Legacy mount

In [0]:
dbutils.widgets.text("secret_scope", "default2")
secret_scope = dbutils.widgets.get("secret_scope")

dbutils.secrets.list(secret_scope)

In [0]:
tenant_id = dbutils.secrets.get(secret_scope, "tenant-id")
client_id = dbutils.secrets.get(secret_scope, "sp-databricks-adls-appid")
secret = dbutils.secrets.get(secret_scope, "sp-databricks-adls-appkey")

container = 'trezio2005'
storage_account = 'dlspl21databricks'

configs = {
  "fs.azure.account.auth.type": "OAuth",
  "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
  "fs.azure.account.oauth2.client.id": client_id,
  "fs.azure.account.oauth2.client.secret": secret,
  "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

mount_point = f"/mnt/{container}_legacy"
source_uri = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"

try:
    dbutils.fs.mount(
        source = source_uri,
        mount_point = mount_point,
        extra_configs = configs
    )
except Exception as e:
    if "Directory already mounted" not in str(e):
        raise e
    

Legacy mount is prohibited on shared clusters. Since this method becomes deprecated we use external locations instead

Alternate method

In [0]:
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
display(dbutils.fs.ls(base_path))